# AI Surveillance & Behavior Analysis System

This notebook contains the end-to-end implementation of an AI Surveillance system that combines **YOLOv8** for human detection and **MobileNetV2** for behavior classification (Normal vs. Suspicious).

## Project Overview
1. **Data Preprocessing**: Extracting human crops from video datasets.
2. **YOLO Fine-tuning**: Training a specialized human detector.
3. **Behavior Classification**: Training a MobileNetV2 model to identify suspicious activities.
4. **Real-time Pipeline**: Integrating both models for live inference.
5. **Interactive Dashboard**: A Streamlit interface for visualization.

## 1. Setup and Environment
Install the necessary libraries for deep learning and computer vision.

In [ ]:
!pip install ultralytics torch torchvision opencv-python pillow tqdm streamlit pandas matplotlib seaborn scikit-learn

## 2. Data Preprocessing: Human Cropping
This stage involves iterating through video files, detecting humans using a base YOLO model, and saving those crops as training data for our behavior classifier.

In [ ]:
import cv2
import os
import torch
from ultralytics import YOLO
from PIL import Image
from tqdm import tqdm

# --- Configuration ---
BASE_DIR = r'c:\Users\sadam\OneDrive\Documents\Field project\archive\dataset-video-split'
OUTPUT_DIR = r'c:\Users\sadam\OneDrive\Documents\Field project\processed_data_cropped'
YOLO_MODEL = 'yolov8n.pt' 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAMES_PER_VIDEO = 15
RESIZE_DIM = (224, 224)

# Label Mapping
ANOMALOUS = ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
NORMAL = ['Clapping', 'Meet_and_Split', 'Normal_Videos', 'Sitting', 'Standing_Still', 'Walking', 'Walking_While_Reading_Book', 'Walking_While_Using_Phone']

def get_label(filename):
    for p in ANOMALOUS: 
        if filename.startswith(p): return "Anomaly"
    for p in NORMAL: 
        if filename.startswith(p): return "Normal"
    return None

def extract_crops(video_path, out_path, n_frames, model):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: return 0
    
    step = max(total // n_frames, 1)
    count = 0
    
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret: break
            
        results = model(frame, device=DEVICE, verbose=False, conf=0.4)
        for result in results:
            for idx, box in enumerate(result.boxes):
                if int(box.cls) == 0: # Person class
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    crop = frame[y1:y2, x1:x2]
                    if crop.size == 0: continue
                    
                    crop = cv2.resize(crop, RESIZE_DIM)
                    name = f"{os.path.basename(video_path).split('.')[0]}_f{i}_p{idx}.jpg"
                    cv2.imwrite(os.path.join(out_path, name), crop)
                    count += 1
    cap.release()
    return count

def run_extraction():
    model = YOLO(YOLO_MODEL)
    for split in ['train', 'valid', 'test']:
        src = os.path.join(BASE_DIR, split)
        if not os.path.exists(src): continue
        
        os.makedirs(os.path.join(OUTPUT_DIR, split, '1'), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, split, '0'), exist_ok=True)
        
        videos = [f for f in os.listdir(src) if f.endswith('.mp4')]
        for v in tqdm(videos, desc=f"Processing {split}"):
            lbl = get_label(v)
            if not lbl: continue
            dst = os.path.join(OUTPUT_DIR, split, '1' if lbl == 'Anomaly' else '0')
            extract_crops(os.path.join(src, v), dst, FRAMES_PER_VIDEO, model)

# Uncomment to run
# run_extraction()

## 3. YOLOv8 Fine-tuning
Fine-tuning YOLOv8 on our specific environment to improve human detection accuracy.

In [ ]:
from ultralytics import YOLO

DATA_YAML = r'c:\Users\sadam\OneDrive\Documents\Field project\data.yaml'

def train_yolo():
    model = YOLO('yolov8n.pt') 
    model.train(
        data=DATA_YAML,
        epochs=10,
        imgsz=224,
        batch=16,
        device='cuda',
        project='yolo_runs',
        name='human_detection'
    )

# Uncomment to run
# train_yolo()

## 4. Behavior Classification: MobileNetV2
Training a binary classifier on the human crops to distinguish between Normal and Suspicious behavior.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

DATA_DIR = r'c:\Users\sadam\OneDrive\Documents\Field project\processed_data_cropped'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_classifier():
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform)
    val_set = datasets.ImageFolder(os.path.join(DATA_DIR, 'valid'), transform)
    
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)

    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.last_channel, 2)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(10):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1} complete.")
    
    torch.save(model.state_dict(), 'best_model.pth')

# Uncomment to run
# train_classifier()

## 5. Real-Time Surveillance Pipeline
Integrating the detection and classification models into a single inference loop for live video.

In [ ]:
import cv2
from ultralytics import YOLO
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

YOLO_PATH = r'c:\Users\sadam\OneDrive\Documents\Field project\runs\detect\yolo_runs\human_detection2\weights\best.pt'
CLASS_PATH = r'c:\Users\sadam\OneDrive\Documents\Field project\best_pretrained_mobilenetv2.pth'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_live():
    # Load models
    det_model = YOLO(YOLO_PATH)
    class_model = models.mobilenet_v2(weights=None)
    class_model.classifier[1] = nn.Linear(class_model.last_channel, 2)
    class_model.load_state_dict(torch.load(CLASS_PATH, map_location=DEVICE))
    class_model = class_model.to(DEVICE).eval()

    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    cap = cv2.VideoCapture(0)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        results = det_model(frame, device=DEVICE, verbose=False, conf=0.5)
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0: continue
                
                img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                tensor = preprocess(img).unsqueeze(0).to(DEVICE)
                
                with torch.no_grad():
                    out = class_model(tensor)
                    prob = torch.softmax(out, dim=1)[0][1].item()
                
                label = "SUSPICIOUS" if prob > 0.85 else "Normal"
                color = (0, 0, 255) if label == "SUSPICIOUS" else (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"{label} {prob:.1%}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        cv2.imshow("Surveillance", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to run
# run_live()

## 6. Streamlit Dashboard
Run the following cell to generate the `dashboard.py` file, which you can then run using `streamlit run dashboard.py`.

In [ ]:
%%writefile dashboard.py
import streamlit as st
import cv2
import torch
from ultralytics import YOLO
from torchvision import models, transforms
from PIL import Image
import torch.nn as nn

st.set_page_config(page_title="AI Surveillance Dashboard", layout="wide")
st.title("🛡️ AI Surveillance & Behavior Analysis")

# Model loading logic here...
st.info("Dashboard code saved to dashboard.py. Run 'streamlit run dashboard.py' to launch.")

## 7. Push to GitHub
To upload this project to your GitHub repository, run the following commands in your terminal:

```bash
git remote add origin https://github.com/sadamushasri22-maker/<your-repo-name>.git
git branch -M main
git push -u origin main
```